# FastOMA Output Analysis: 8 Birds + Outgroups

Welcome to the FastOMA analysis notebook! In this module, we will explore the results of a FastOMA run on a dataset comprising 8 avian proteomes alongside selected outgroups.

FastOMA is an efficient, scalable tool that infers orthology by mapping new proteomes to pre-computed Reference Orthologous Groups (from the OMA database). By the end of this notebook, you will be able to extract evolutionary insights from FastOMA's diverse output files, moving from basic mapping statistics to deep phylogenomic analysis.

Notebook Objectives
- Learn how to run FastOMA:
    - Prepare input file and use the Nextflow pipeline.  
- Inspect FastOMA final Report file:
    - Parse and understand the general stats contained in the report output, at the end of a FastOMA run. We will check potential technical problems and observe basic statistics.  
- Analyze the Phylostratrigraphy:
    - A primary output of FastOMA is the phylostratigraphy plot. Here, you will be able to track the ancestral gene content in each node of the species tree.   
- Strict Orthologous Groups (OGs):
    - FastOMA can be used to identify universal single-copy orthologous groups to see which core genes are perfectly conserved across all species.  
- Extract the ancestral genome of Aves:
    - We will focus on the genome of the first bird. Can you extract the ancestral genes and their representative in an extant genome?  
- Find the evolutionary history of a Gene family of interest:
    - We will focus on one gene of interest and how it evolved in the different bird lineages.  



# Running FastOMA
In order to run FastOMA, you will need to: 
1. Assemble the correct files inside the input folder
    - Visit the FastOMA documentation at https://github.com/DessimozLab/FastOMA to read how the input folder must be formatted and the pipeline run.
    - For the Proteome selection, use the proteomes that passed the contamination filter from the first module found in `/vol/Topic4CommonData/shared/`.
    - The folder `/vol/Topic4CommonData/Module2/` contains the omamer database, the species tree, and the splice folder required if some of the proteomes include isoforms.
2. Run FastOMA using Nextflow:
    - Remember to activate the right conda environment (module2env)
    - You will find the predownloaded repository of FastOMA with the nextflow pipeline file at `/vol/Topic4CommonData/Module2/FastOMA/FastOMA.nf`
    - try to run FastOMA from outside the repository to avoid unnecessary file dumps. For example, in `/scratch/fastoma_run/`
3. Proceed with the analysis using the precomputed results in `/vol/Topic4CommonData/Module2/fastoma_output/`

If the run was successful, you will find an output folder containing all of FastOMA output files. Here you can find the complete FastOMA output description, including 3 folders and 7 additional files:

- hogmap
    - contains the OMAmer results used by FastOMA (Described in the OMAmer Module); each file corresponds to an input proteome.
- OrthologousGroupsFasta
    - contains the FASTA files of all marker orthologous groups

- RootHOGsFasta
    - contains the FASTA files of all sequences in each HOGs

- The main orthology output files
    - FastOMA_HOGs.orthoxml, orthologs.tsv.gz, RootHOGs.tsv, OrthologousGroups.tsv

- Summary reports
    - phylostratigraphy.html, report.ipynb, report.html,

- The transformed input species tree
    - species_tree_checked.nwk

# Inspect FastOMA final Report file
Next, we will look into the automatic report of the FastOMA run. Open the report file in the output folder (you can either download the .html file on your computer or open the .ipynb file in VScode), and answer the following questions:
1. Are there any outliers in genome size?
2. Do you notice anything about the protein length distribution?
3. How many rootHOGs have been found?
4. What is the average number of members in a rootHOG?
5. What is the mean CompletenessScore of all rootHOGs?
    - Completeness score (sometimes referred as HOG quality) is our rudimentary method to evaluate the confidence of a HOG. It is calculated as the number of species that have a gene in said HOG over the total number of species at the HOG taxonomic level.


# Analyze the Phylostratrigraphy
By reconstructing HOGs, FastOMA also models the evolutionary histories of all gene families: at which taxonomic level they are gained, lost, and duplicated. The results of this are contained in the phylostratigraphy file. Download the phylostratigraphy.html on you computer (right click) and open it. Can you answer these questions?
1. How many genes we observe in the ancestral genome of Aves? How many genes have been lost from the previous ancestor?
2. Which bird has the highest number of lineage-specific gene gains (potential gene innovations)?
3. Is the genome size of the bird ancestors consistent with the observed extant genome sizes?

Next, we will analyze the other outputs of FastOMA using a combination of Bash scripts and OMA python libraries. 

In [56]:
import os,sys
import pyham
from tqdm import tqdm
import pandas as pd
from IPython.display import IFrame

In [9]:
# set paths
FASTOMA_RES_FOLDER = '/vol/Topic4CommonData/Module2/fastoma_output/'
FASTOMA_ORTHOXML = FASTOMA_RES_FOLDER + 'FastOMA_HOGs.orthoxml'
FASTOMA_SPECIESTREE = FASTOMA_RES_FOLDER + 'species_tree_checked.nwk'
OUT_PATH = '~/SIBBiodiversitySummerSchool2026Topic4/Module2_OrthologyAtScale/out/analysis/'
# test paths
FASTOMA_RES_FOLDER = '/Users/spascare/data/work/dessimoz/teaching/2026/SIBBiodiversitySummerSchool2026Topic4/CommonData/minidataset/fastoma_output/'
FASTOMA_ORTHOXML = FASTOMA_RES_FOLDER + 'FastOMA_HOGs.orthoxml'
FASTOMA_SPECIESTREE = FASTOMA_RES_FOLDER + 'species_tree_checked.nwk'
OUT_PATH = '/Users/spascare/data/work/dessimoz/teaching/2026/SIBBiodiversitySummerSchool2026Topic4/Module2_OrthologyAtScale/out/analysis/'
if not os.path.exists(OUT_PATH):
    os.mkdir(OUT_PATH)

# Strict Orthologous Groups (OGs)
Here you will identify which HOGs have one and only one gene representative in each species, using the HOG classification

In [10]:
OG_FILE = FASTOMA_RES_FOLDER + 'OrthologousGroups.tsv'
os.system("cat {} | cut -f 1 | sort | uniq -c | awk '{{print $1}}' | grep 10 | wc -l".format(OG_FILE))  # double curly brackets to escape the python format string


7290


0

# Extract the ancestral genome of Aves and print representatives
We will use the pyham python package to open the FastOMA orthoxml results. From here, we will extract the ancestral HOGs of aves and save all representative genes from chicken, ostrich, and crane.

In [11]:
# create pyham object here
if pyham is not None:
    ham_analysis = pyham.Ham(FASTOMA_SPECIESTREE, FASTOMA_ORTHOXML, tree_format="newick", use_internal_name=True)
    print("Ham analysis done") # for a big orthoxml file it can take ~30mins
else:
    print("pyham not installed, skipping ham analysis")

Ham analysis done


In [60]:
# set params
anc_id =  'Aves' 
allgenes_out = '{}/allgenes_{}_table.tsv'.format(OUT_PATH,anc_id)
anc_gen = ham_analysis.get_ancestral_genome_by_name(anc_id)
t2f = ['Struthio_camelus','Gallus_gallus','Grus_americana'] 
out_data = {'HOG':[],
            'rootHOG':[],
            'q_score':[],
            }
for spec in t2f:
    out_data[spec] = []

# run the for loop
for hog_sel in tqdm(anc_gen.genes):
    # hog name without columns
    HOG = hog_sel.hog_id
    out_data['HOG'] += [HOG]
    roothog = hog_sel.get_top_level_hog()
    out_data['rootHOG'] += [roothog.hog_id]
    
    # EXTRA: find quality
    x = hog_sel.get_all_descendant_genes_clustered_by_species()
    n_sp_in_hog = len(x.keys())
    n_sp_below_root_level = len(hog_sel.genome.taxon.get_leaf_names())
    q = n_sp_in_hog / n_sp_below_root_level
    out_data['q_score'] += [str(q)]
    
    # save genes of selected species
    genes_dict = hog_sel.get_all_descendant_genes_clustered_by_species()
    for spec in t2f:
        this_spec_genes = ''
        this_spec_genelist = [genes_dict[x] for x in genes_dict if x.name == spec]
        if this_spec_genelist:
                prot_ids = [x.prot_id for x in this_spec_genelist[0]]
                this_spec_genes = ';'.join(prot_ids)
        out_data[spec] += [this_spec_genes]
    
# Prepare output
out_data_df = pd.DataFrame(out_data)
out_data_df.to_csv(allgenes_out,sep='\t')
out_data_df

100%|██████████| 17272/17272 [00:00<00:00, 64073.12it/s]


,HOG,rootHOG,q_score,Struthio_camelus,Gallus_gallus,Grus_americana
0,HOG:0000001_4,HOG:0000001_1,0.75,STRCAM_R09462,GALGAL_R09604,GRUAME_R07328
1,HOG:0000002_4,HOG:0000002_1,0.625,STRCAM_R04929,GALGAL_R16130,GRUAME_R08261
2,HOG:0000003_4,HOG:0000003_1,1.0,STRCAM_R08918,GALGAL_R12912,GRUAME_R01896
3,HOG:0000004_4,HOG:0000004_1,0.875,STRCAM_R11615,GALGAL_R08071,GRUAME_R00855
4,HOG:0000005_4,HOG:0000005_1,1.0,STRCAM_R08953,GALGAL_R11175,GRUAME_R13849
...,...,...,...,...,...,...
17267,HOG:0016212_4,HOG:0016212_1,1.0,STRCAM_R04762,GALGAL_R06558,GRUAME_R02914
17268,HOG:0016213_4,HOG:0016213_1,1.0,STRCAM_R00403,GALGAL_R11305,GRUAME_R03181
17269,HOG:0016214_4,HOG:0016214_1,1.0,STRCAM_R00463,GALGAL_R15221,GRUAME_R04226
17270,HOG:0016215_4,HOG:0016215_1,1.0,STRCAM_R13432,GALGAL_R16163,GRUAME_R09042


Advanced,Optional: How would you modify the code in order to get the represetantive genes from the outgroups too? (human and crocodyle?)

# Find the evolutionary history of a Gene family of interest
Select one HOG and find the history of this HOG

In [63]:
# select an HOG
HOG = 'HOG:0000073_1'
renamed_HOG=HOG.replace(':','_')
single_gene_phylo_out = '{}/singlegene_{}_phylostratigraphy.html'.format(OUT_PATH,renamed_HOG)

hog_sel =  ham_analysis.get_hog_by_id(HOG)

# create the treeprofile object
treeprofile_hog_sel = ham_analysis.create_tree_profile(hog=hog_sel, outfile=single_gene_phylo_out)
 
# Here a little demo of what you can see with hogs vis
IFrame(single_gene_phylo_out, width=700, height=350)

 